# 🚗 Analyse Statistique Avancée des Accidents de la Route aux États-Unis


## Importer data

In [ ]:
import pandas as pd
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

#Importer data
df= pd.read_csv("us.csv")

print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500000 entries, 0 to 499999
Data columns (total 46 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   ID                     500000 non-null  object 
 1   Source                 500000 non-null  object 
 2   Severity               500000 non-null  int64  
 3   Start_Time             500000 non-null  object 
 4   End_Time               500000 non-null  object 
 5   Start_Lat              500000 non-null  float64
 6   Start_Lng              500000 non-null  float64
 7   End_Lat                0 non-null       float64
 8   End_Lng                0 non-null       float64
 9   Distance(mi)           500000 non-null  float64
 10  Description            500000 non-null  object 
 11  Street                 500000 non-null  object 
 12  City                   499978 non-null  object 
 13  County                 500000 non-null  object 
 14  State                  500000 non-nu

## Nettoyage

In [ ]:
# Conversion des dates 'Start_time'
df['Start_Time'] = pd.to_datetime(df['Start_Time'], errors='coerce')

# Conversion des dates 'End_time'
df['End_Time'] = pd.to_datetime(df['End_Time'], errors='coerce')

# Conversion des dates 'Weather_Timestamp '
df['Weather_Timestamp'] = pd.to_datetime(df['Weather_Timestamp'], errors='coerce')

# Supprimer lees colonne vide 'End_Lat', 'End_Lng'
df.drop(['End_Lat', 'End_Lng'], axis=1, inplace=True)

# Supprimer les colonnes sans valeur analytique
df.drop(['Country', 'Description'], axis=1, inplace=True)

#Gestion des valeurs manquantes 
twilight_cols = [
    'City',
    'Zipcode',
    'Timezone',
    'Weather_Condition',
    'Wind_Direction',
    'Sunrise_Sunset',
    'Civil_Twilight',
    'Nautical_Twilight',
    'Astronomical_Twilight',
]
for col in twilight_cols:
    df[col].fillna('Unknown', inplace=True)


# Colonnes météo numériques → median
weather_num_cols = [
    'Temperature(F)', 'Humidity(%)',
    'Pressure(in)', 'Visibility(mi)',
    'Wind_Speed(mph)'
]

for col in weather_num_cols:
    df[col] = df.groupby(
        ['City', 'Airport_Code','Start_Time','County']
    )[col].transform(lambda x: x.fillna(x.median()))

for col in weather_num_cols:
    df[col].fillna(df[col].median(), inplace=True)

# Colonnes météo avec beaucoup de valeurs manquantes
df['Wind_Chill(F)'].fillna(0, inplace=True)
df['Precipitation(in)'].fillna(0, inplace=True)


print(f"Données chargées : {df.shape[0]:,} lignes et {df.shape[1]} colonnes.")
df.info()
df.head()


Données chargées : 500,000 lignes et 44 colonnes.


In [ ]:
df['Start_Time'] = pd.to_datetime(df['Start_Time'], errors='coerce')

# Conversion des dates 'End_time'
df['End_Time'] = pd.to_datetime(df['End_Time'], errors='coerce')

# Conversion des dates 'Weather_Timestamp '
df['Weather_Timestamp'] = pd.to_datetime(df['Weather_Timestamp'], errors='coerce')

# Supprimer lees colonne vide 'End_Lat', 'End_Lng'
df.drop(['End_Lat', 'End_Lng'], axis=1, inplace=True)

# Supprimer les colonnes sans valeur analytique
df.drop(['Country', 'Description'], axis=1, inplace=True)

#Gestion des valeurs manquantes 
twilight_cols = [
    'City',
    'Zipcode',
    'Timezone',
    'Airport_Code',
    'Weather_Condition',
    'Wind_Direction',
    'Sunrise_Sunset',
    'Civil_Twilight',
    'Nautical_Twilight',
    'Astronomical_Twilight',
]
for col in twilight_cols:
    df[col] = df[col].fillna('Unknown')


# Colonnes météo numériques → median
weather_num_cols = [
    'Temperature(F)', 'Humidity(%)',
    'Pressure(in)', 'Visibility(mi)',
    'Wind_Speed(mph)'
]

for col in weather_num_cols:
    df[col].fillna(df[col].median(), inplace=True)


# 1. Sélection des colonnes météo corrélées
weather_cols = ['Temperature(F)', 'Humidity(%)', 'Pressure(in)', 'Visibility(mi)', 'Wind_Speed(mph)']

# 2. Initialisation de l'imputer (utilise la régression bayésienne par défaut)
imputer = IterativeImputer(max_iter=10, random_state=42)

# 3. Application de l'imputation
# On l'applique sur les colonnes numériques
df[weather_cols] = imputer.fit_transform(df[weather_cols])



# Colonnes météo avec beaucoup de valeurs manquantes
df['Wind_Chill(F)'].fillna(0, inplace=True)
df['Precipitation(in)'].fillna(0, inplace=True)

# Trier par temps pour s'assurer que le forward fill est logique
df = df.sort_values(by='Start_Time')

# Remplir les Timestamps manquants par la valeur précédente
df['Weather_Timestamp'] = df['Weather_Timestamp'].ffill()

print(f"Données chargées : {df.shape[0]:,} lignes et {df.shape[1]} colonnes.")
df.info()